# EEGNet — bezobsługowy trening w Google Colab

1. Wybierz **Środowisko wykonawcze → Zmień typ środowiska wykonawczego → GPU**.
2. Uruchom montowanie Drive i zaakceptuj dostęp.
3. Ustaw konfigurację, uruchom ostatnią komórkę i możesz zostawić trening.

Checkpoint, historia i log są zapisywane na Google Drive po każdej epoce. Po przerwaniu sesji uruchom notebook ponownie — trening wznowi się automatycznie.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Konfiguracja eksperymentu
GITHUB_REPO = 'https://github.com/taf4you2/biai.git'
GITHUB_BRANCH = 'codex/eeg-pipeline-qc'

# Notebook sam znajdzie także plik z dopiskiem typu „(1)”.
DRIVE_DATA_DIR = '/content/drive/MyDrive/biai/data'
ZIP_PATTERN = 'biai_eeg_qc_0_0p8*.zip'

TEST_PARTICIPANT = 'mole'
FULL_EPOCHS = 30
BATCH_SIZE = 256
RUN_NAME = f'eegnet_{TEST_PARTICIPANT}_{FULL_EPOCHS}ep_colab'
DRIVE_RESULTS = f'/content/drive/MyDrive/biai/results/{RUN_NAME}'

# Ustaw True tylko wtedy, gdy chcesz skasować checkpoint i zacząć od początku.
FORCE_RETRAIN = False

In [ ]:
# TRYB BEZOBSŁUGOWY — uruchom tę jedną komórkę i zostaw ją pracującą.
from datetime import datetime
from pathlib import Path
import json
import shutil
import subprocess
import zipfile

import matplotlib.pyplot as plt
import pandas as pd
import torch


def run_and_log(command, log_path, cwd=None):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a', encoding='utf-8') as log:
        header = f"\n[{datetime.now().isoformat(timespec='seconds')}] {' '.join(map(str, command))}\n"
        print(header, end='')
        log.write(header)
        log.flush()
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        return_code = process.wait()
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, command)


if not torch.cuda.is_available():
    raise RuntimeError('GPU nie jest aktywne. Zmień typ środowiska wykonawczego na GPU.')
print('GPU:', torch.cuda.get_device_name(0))

repo_dir = Path('/content/biai')
local_zip = Path('/content/biai_eeg_qc_0_0p8.zip')
results_dir = Path(DRIVE_RESULTS)

if FORCE_RETRAIN and results_dir.exists():
    shutil.rmtree(results_dir)
results_dir.mkdir(parents=True, exist_ok=True)
log_path = results_dir / 'training.log'

# Świeża kopia kodu przy każdym uruchomieniu sesji.
if repo_dir.exists():
    shutil.rmtree(repo_dir)
run_and_log(
    ['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH, GITHUB_REPO, str(repo_dir)],
    log_path,
)

# Automatyczne znalezienie ZIP-a na Drive.
data_dir = Path(DRIVE_DATA_DIR)
zip_matches = sorted(data_dir.glob(ZIP_PATTERN)) if data_dir.is_dir() else []
if not zip_matches:
    zip_matches = sorted(Path('/content/drive/MyDrive').rglob(ZIP_PATTERN))
if not zip_matches:
    raise FileNotFoundError(f'Nie znaleziono {ZIP_PATTERN} na Google Drive')
drive_zip = zip_matches[0]
print('Paczka danych:', drive_zip)
shutil.copy2(drive_zip, local_zip)

with zipfile.ZipFile(local_zip) as archive:
    archive.extractall(repo_dir)
dataset_dir = repo_dir / 'event_epoch_multisession_image_on_0_0p8_qc'

run_and_log(
    ['python', '-u', str(repo_dir / 'scripts/verify_colab_dataset.py'), '--dataset-dir', str(dataset_dir)],
    log_path,
    cwd=repo_dir,
)

# Pełny trening. --resume wczyta checkpoint z Drive, jeżeli poprzednia sesja została przerwana.
train_command = [
    'python', '-u', str(repo_dir / 'scripts/train_eegnet.py'),
    '--dataset-dir', str(dataset_dir),
    '--output-dir', str(results_dir),
    '--split', 'participant_image',
    '--test-participant', TEST_PARTICIPANT,
    '--val-split', 'participant',
    '--normalization', 'participant',
    '--balanced-sampler', 'category_participant',
    '--only-qc-accepted',
    '--epochs', str(FULL_EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--checkpoint-every', '1',
    '--resume',
]
run_and_log(train_command, log_path, cwd=repo_dir)

# Automatyczne podsumowanie i wykresy zapisane obok modelu.
summary_path = results_dir / 'eegnet_summary.json'
history_path = results_dir / 'eegnet_history.csv'
with summary_path.open(encoding='utf-8') as handle:
    summary = json.load(handle)
history = pd.read_csv(history_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history.plot(x='epoch', y=['train_loss', 'val_loss'], ax=axes[0], grid=True, title='Loss')
history.plot(x='epoch', y='val_acc', ax=axes[1], grid=True, title='Validation accuracy')
fig.tight_layout()
fig.savefig(results_dir / 'training_curves.png', dpi=160)
plt.show()

print('\nTRENING ZAKOŃCZONY')
print('Najlepsza epoka:', summary['best_epoch'])
print('Validation accuracy:', f"{summary['best_validation_accuracy']:.2%}")
print('Test accuracy:', f"{summary['final_test_accuracy']:.2%}")
print('Wyniki:', results_dir)